# Recogida con ruido

Aquest notebook genera la mostra inicial del treball: 1000 colors RGB aleatoris, el seu chroma i les imatges PNG que despres passarem als models.

La logica important esta a `scripts/utils.py`. Aqui intento deixar nomes la configuracio i les crides principals, perque si no el notebook es fa impossible de seguir.


## Configuracio

Fixem els parametres de l'experiment. La seed es important per poder regenerar exactament la mateixa mostra en un altre ordinador.

In [ ]:
from pathlib import Path
import sys
import time

N_COLORS = 1000
SEED = 23
IMAGE_SIZE = 100
NEAR_DISTANCE = 30

# Aquesta part crida l'API i pot gastar diners. Ho deixo apagat per defecte.
RUN_MODEL_QUERIES = True
MODEL_NAMES = ["gpt-4o", "gpt-4o-mini"]
MODEL_TEMPERATURE = 0.2
MODEL_MAX_IMAGES = None  # posa un numero petit, per exemple 5, si vols fer una prova rapida



def _format_duration_local(seconds):
    seconds = max(0, float(seconds))
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes, remaining_seconds = divmod(seconds, 60)
    if minutes < 60:
        return f"{int(minutes)}m {remaining_seconds:04.1f}s"
    hours, remaining_minutes = divmod(minutes, 60)
    return f"{int(hours)}h {int(remaining_minutes)}m {remaining_seconds:04.1f}s"

def _install_cell_timer():
    ipython = get_ipython()
    if ipython is None or getattr(ipython, "_color_llm_timer_installed", False):
        return

    def _start_cell_timer(info):
        ipython.user_ns["_cell_started_at"] = time.perf_counter()

    def _stop_cell_timer(result):
        started_at = ipython.user_ns.get("_cell_started_at")
        if started_at is not None:
            elapsed = time.perf_counter() - started_at
            print(f"Temps cel.la: {_format_duration_local(elapsed)}")

    ipython.events.register("pre_run_cell", _start_cell_timer)
    ipython.events.register("post_run_cell", _stop_cell_timer)
    ipython._color_llm_timer_installed = True

_install_cell_timer()

# Canvia a True nomes si vols tornar a generar la mostra des de zero.
REGENERATE_SAMPLE = False

# Casella de seguretat: si es posa a True, esborra CSVs, imatges PNG i logs generats.
DELETE_EXISTING_SAMPLE = False

base_dir = Path("recollida-dades")
if not base_dir.exists():
    base_dir = Path("..").resolve()
if not (base_dir / "scripts").exists():
    raise FileNotFoundError(f"No trobo recollida-dades/scripts des de {Path.cwd()}")
experiment_dir = base_dir / "experiments" / "soroll"

images_dir = experiment_dir / "images"
tmp_dir = base_dir / "archive" / "tmp" / "soroll-notebook"
logs_dir = experiment_dir / "logs"
scripts_dir = base_dir / "scripts"

for folder in [csv_dir, images_dir, tmp_dir, logs_dir]:
    folder.mkdir(parents=True, exist_ok=True)

if str(scripts_dir) not in sys.path:
    sys.path.append(str(scripts_dir))

print("Carpeta base:", base_dir)


## Carrega de scripts

Importem les funcions del script. Si alguna no existeix, parem aqui, perque probablement el notebook no esta trobant be `scripts/utils.py`.

In [ ]:
import importlib
import pandas as pd
from IPython.display import Image as DisplayImage, display

import utils
importlib.reload(utils)

format_duration = utils.format_duration
build_final_sample = utils.build_final_sample
collect_model_outputs = utils.collect_model_outputs
delete_sample_outputs = utils.delete_sample_outputs
delete_png_files = utils.delete_png_files
generate_images_from_sample = utils.generate_images_from_sample
generate_unique_colors = utils.generate_unique_colors
save_csv = utils.save_csv
save_rgb_sample_grid = utils.save_rgb_sample_grid
save_rgb_sample_map = utils.save_rgb_sample_map
save_chroma_distribution = utils.save_chroma_distribution
write_log = utils.write_log

required_functions = [
    "format_duration",
    "generate_unique_colors",
    "generate_images_from_sample",
    "save_rgb_sample_grid",
    "save_rgb_sample_map",
    "save_chroma_distribution",
    "delete_sample_outputs",
    "delete_png_files",
    "save_csv",
    "write_log",
]
missing = [name for name in required_functions if name not in globals()]

if missing:
    raise RuntimeError(f"Error: scripts no cargados correctamente: {missing}")

if DELETE_EXISTING_SAMPLE:
    removed = delete_sample_outputs(csv_dir=csv_dir, images_dir=images_dir, logs_dir=logs_dir)
    print(f"Fitxers esborrats: {len(removed)}")

write_log("Notebook de recollida inicialitzat", logs_dir / "pipeline.log")


## Generacio o carrega de la mostra de colors

Si `input_image_sample.csv` ja existeix, el carreguem i no el tornem a generar. Aixi evitem canviar la mostra sense voler.

Per regenerar-ho tot, cal posar `DELETE_EXISTING_SAMPLE = True` i `REGENERATE_SAMPLE = True` a la configuracio. Ho deixo aixi una mica explicit per no liar-la per accident.


In [ ]:
input_path = csv_dir / "input_image_sample.csv"

if input_path.exists() and not REGENERATE_SAMPLE:
    input_sample = pd.read_csv(input_path)
    print(f"Mostra carregada: {input_path}")
else:
    input_sample = generate_unique_colors(
        n=N_COLORS,
        seed=SEED,
        progress=True,
        log_file=logs_dir / "pipeline.log",
    )
    save_csv(input_sample, input_path)
    write_log(f"Mostra generada: {input_path}", logs_dir / "pipeline.log")
    print(f"Mostra generada: {input_path}")

input_sample.head()


## Mapa de color i comprovacio rapida de biaix

Aqui genero tres visualitzacions. No son una prova matematica perfecta, pero ajuden bastant a veure si la mostra fa coses rares.

- **Graella de colors**: mostra els 1000 colors exactes ordenats per to. Serveix per mirar la mostra a ull, no per demostrar uniformitat.
- **Distribucio per canal RGB**: son tres histogrames, un per R, G i B. Com que els RGB surten aleatoris, les barres haurien de quedar mes o menys equilibrades, pero no identiques.
- **Hue vs saturation**: posa el to a l'eix X i la saturacio a l'eix Y. Els punts de baix son colors mes grisos. Important: una mostra uniforme en RGB no te per que ser uniforme en HSV.
- **R vs G, R vs B i G vs B**: son projeccions del cub RGB. El que volem veure es que els punts omplen el quadrat sense forats grans ni patrons massa clars.
- **max RGB vs min RGB**: compara el canal mes alt amb el mes baix de cada color. La forma triangular es normal, perque el minim mai pot ser mes gran que el maxim.
- **Distribucio del chroma**: chroma baix vol dir color proper al gris, i chroma alt vol dir color mes viu o saturat. Aquesta distribucio tampoc ha de ser plana obligatoriament, perque depen de com RGB es transforma a Lab.

Els punts es pinten amb el seu color real, sense contorns blancs o negres, per no confondre la visualitzacio. Si surt algun punt quasi negre, es perque aquell color realment es fosc.


In [ ]:
color_grid_path = tmp_dir / "rgb_sample_grid.png"
color_map_path = tmp_dir / "rgb_sample_map.png"
chroma_plot_path = tmp_dir / "rgb_chroma_distribution.png"

# Les visualitzacions son barates de generar, aixi que les fem sempre a partir del CSV actual.
# D'aquesta manera no ens quedem mirant una imatge antiga del notebook.
save_rgb_sample_grid(input_sample, color_grid_path)
save_rgb_sample_map(input_sample, color_map_path)
save_chroma_distribution(input_sample, chroma_plot_path)
write_log(f"Visualitzacions RGB generades a tmp: {color_grid_path}, {color_map_path}, {chroma_plot_path}", logs_dir / "pipeline.log")

print(f"Graella generada: {color_grid_path}")
print(f"Mapa de diagnosi generat: {color_map_path}")
print(f"Grafic de chroma generat: {chroma_plot_path}")

print("Graella de les 1000 mostres:")
display(DisplayImage(filename=str(color_grid_path)))

print("Mapa de diagnosi de distribucio:")
display(DisplayImage(filename=str(color_map_path)))

print("Distribucio del chroma:")
display(DisplayImage(filename=str(chroma_plot_path)))

input_sample[["r", "g", "b", "chroma"]].describe()


## Generacio de les imatges

Les imatges individuals tambe es reutilitzen si ja existeixen. Si falten o si activem `REGENERATE_SAMPLE`, primer esborrem els PNG antics de `images/` i despres es tornen a crear. Aquest pas mostra barra de progres amb ETA per no anar a cegues.

Cada imatge es de `IMAGE_SIZE x IMAGE_SIZE` pixels. La composicio es: 60% de pixels exactament del color objectiu, 20% de pixels amb colors propers pero aleatoris entre ells, i el 20% restant amb colors aleatoris de tot l'espectre RGB. Les guardem en PNG per no modificar el color amb compressio JPEG.


In [ ]:
expected_images = {name for name in input_sample["image_name"]}
existing_images = {path.name for path in images_dir.glob("*.png")}
missing_images = expected_images - existing_images

if missing_images or REGENERATE_SAMPLE:
    removed_images = delete_png_files(images_dir)
    if removed_images:
        print(f"Imatges antigues esborrades: {len(removed_images)}")

    image_paths = generate_images_from_sample(
        input_sample,
        output_dir=images_dir,
        size=IMAGE_SIZE,
        near_distance=NEAR_DISTANCE,
        seed=SEED,
        progress=True,
        log_file=logs_dir / "pipeline.log",
    )
    write_log(f"Imatges antigues esborrades: {len(removed_images)}", logs_dir / "pipeline.log")
    write_log(f"Imatges generades: {len(image_paths)}", logs_dir / "pipeline.log")
    print(f"Imatges generades: {len(image_paths)}")
else:
    image_paths = sorted(images_dir.glob("*.png"))
    print(f"Imatges ja existents: {len(image_paths)}")

len(image_paths), image_paths[:3]


In [ ]:
from pathlib import Path
images_path = Path("images")
num_files = len(list(images_path.glob("*.png")))
print('Conteig imatges observades:',num_files)

## Consulta als models

Aquesta part ja queda preparada per executar el benchmark amb models de visio. La clau no s'ha d'escriure mai al notebook: s'ha de posar en un fitxer `.env` local amb `OPENAI_API_KEY=...`. Aquest fitxer esta ignorat per Git.

Per evitar ensurts, `RUN_MODEL_QUERIES` esta a `False` per defecte. Quan ho vulgui executar de veritat, el canvio a `True`. Si falla a mig cami, els resultats es van guardant a `experiments/soroll/csv/outputmodel_image_sample_4o.csv`, i la propera execucio salta les parelles imatge/model que ja existeixen. La cel.la tambe mostra una barra de progres amb ETA i escriu detalls a `logs/model_calls.log`.


In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

output_path = csv_dir / "outputmodel_image_sample_4o.csv"
model_log_path = logs_dir / "model_calls.log"

if RUN_MODEL_QUERIES:
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Falta OPENAI_API_KEY. Crea un fitxer .env local amb la clau abans d'executar models.")

    client = OpenAI()
    model_outputs = collect_model_outputs(
        client=client,
        image_paths=image_paths,
        models=MODEL_NAMES,
        temperature=MODEL_TEMPERATURE,
        output_path=output_path,
        log_file=model_log_path,
        max_images=MODEL_MAX_IMAGES,
    )

    print(f"Resultats de models guardats: {output_path}")
    print(f"Files totals: {len(model_outputs)}")
    display(model_outputs.tail())
else:
    if output_path.exists():
        model_outputs = pd.read_csv(output_path)
        print(f"Models no executats ara. Carrego resultats existents: {len(model_outputs)} files")
    else:
        model_outputs = pd.DataFrame()
        print("Models no executats. Posa RUN_MODEL_QUERIES = True quan vulguis cridar l'API.")


In [ ]:
output_path = csv_dir / "outputmodel_image_sample_4o.csv"
final_path = csv_dir / "sample-colors_4o.csv"

if output_path.exists():
    model_outputs = pd.read_csv(output_path)
    final_sample = build_final_sample(input_sample, model_outputs)
    save_csv(final_sample, final_path)
    write_log(f"Dataset final guardat: {final_path} files={len(final_sample)}", logs_dir / "pipeline.log")
    final_sample.head()
else:
    print("Encara no existeix outputmodel_image_sample_4o.csv. Primer cal executar la cel.la de models.")


## Resultats generats

Comprovem que tenim el CSV d'entrada i les imatges. Si aqui surt 1000, la part base de recollida ja esta feta.

In [ ]:
print("CSV generats:", sorted(path.name for path in csv_dir.glob("*.csv")))
print("Nombre d'imatges:", len(list(images_dir.glob("*.png"))))
print("Fitxers temporals:", sorted(path.name for path in tmp_dir.glob("*.png")))
print("Logs:", sorted(path.name for path in logs_dir.iterdir()))

In [ ]:
final_sample

## Analisis preliminar de la muestra

Este apartado resume la calidad de las respuestas de los modelos sobre la muestra final. Es un analisis descriptivo previo al analisis estadistico formal: sirve para entender la escala del error, detectar respuestas repetidas o demasiado genericas, y comprobar si el error cambia segun el chroma del color objetivo.

### Variables y formulas usadas

Cada imagen tiene un color objetivo en RGB:

$$
(R_i, G_i, B_i)
$$

Cada modelo devuelve una estimacion, tambien en RGB:

$$
(\hat{R}_{ij}, \hat{G}_{ij}, \hat{B}_{ij})
$$

para la imagen $i$ y el modelo $j$.

El **error cromatico** se calcula como distancia euclidea en el espacio RGB:

$$
error_{ij} = \sqrt{(\hat{R}_{ij} - R_i)^2 + (\hat{G}_{ij} - G_i)^2 + (\hat{B}_{ij} - B_i)^2}
$$

Un error de 0 significa que el modelo ha devuelto exactamente el mismo color RGB que la imagen. Valores mas altos indican mas distancia cromatica.

El **chroma** resume la intensidad o saturacion del color. Lo calculamos convirtiendo RGB a HSV y usando el componente de saturacion $S_i$:

$$
chroma_i = S_i = \frac{\max(R_i, G_i, B_i) - \min(R_i, G_i, B_i)}{\max(R_i, G_i, B_i)}
$$

si $\max(R_i, G_i, B_i) > 0$. Si el maximo es 0, el color es negro puro y definimos $chroma_i = 0$.

Interpretacion:

- chroma cercano a 0: color grisaceo o poco saturado;
- chroma cercano a 1: color muy saturado;
- el chroma no mide brillo, sino separacion relativa entre canales RGB.

### Que miramos aqui

- Distribucion del error por modelo.
- Media, mediana y desviacion tipica del error.
- Grafico violin + boxplot para comparar forma, mediana y dispersion.
- Lazy response: concentracion de respuestas repetidas y uso de colores demasiado genericos.
- Correlacion entre error y chroma.
- Comportamiento por 4 grupos de chroma, usando cuartiles de la muestra.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image as DisplayImage, display

# Rutas robustas: funciona desde la raiz del repo o desde recollida-dades/notebooks.
_nb_cwd = Path.cwd()
if (_nb_cwd / "recollida-dades").exists():
    _recollida_dir = _nb_cwd / "recollida-dades"
elif _nb_cwd.name == "notebooks" and _nb_cwd.parent.name == "recollida-dades":
    _recollida_dir = _nb_cwd.parent
elif _nb_cwd.name == "recollida-dades":
    _recollida_dir = _nb_cwd
else:
    _recollida_dir = Path("..").resolve()

experiment_dir = _recollida_dir / "experiments" / "soroll"
results_dir = experiment_dir / "results"
plots_dir = experiment_dir / "plots"

final_results_path = results_dir / "sample_colors_tots_models_actual.csv"
error_summary_path = results_dir / "resum_error_tots_models.csv"
lazy_summary_path = results_dir / "resum_lazy_response_tots_models.csv"
accuracy_summary_path = results_dir / "resum_percent_acierto_tots_models.csv"

required_paths = [final_results_path, error_summary_path, lazy_summary_path, accuracy_summary_path]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Faltan resultados finales: " + ", ".join(str(path) for path in missing_paths))

sample_results = pd.read_csv(final_results_path)
error_summary = pd.read_csv(error_summary_path)
lazy_summary = pd.read_csv(lazy_summary_path)
accuracy_summary = pd.read_csv(accuracy_summary_path)

model_order = ["gpt-4o", "gpt-4o-mini", "gpt-5", "gpt-5.1", "gpt-5.2", "gpt-5.4", "gpt-5.5"]
sample_results = sample_results[sample_results["model"].isin(model_order)].copy()
sample_results["model"] = pd.Categorical(sample_results["model"], categories=model_order, ordered=True)

print("Dataset combinado:", final_results_path)
print("Filas:", len(sample_results))
print("Imagenes unicas:", sample_results["image_name"].nunique())
print("Modelos:", list(sample_results["model"].cat.categories))

display(sample_results.groupby("model", observed=False).size().reset_index(name="n"))


### Descriptiva del error cromatico

Comparamos media, mediana y desviacion tipica para detectar tanto el rendimiento medio como la dispersion de cada modelo.

- La **media** resume el error esperado, pero puede subir mucho si hay casos extremos.
- La **mediana** es mas robusta: indica el error tipico de una respuesta central.
- La **desviacion tipica** mide dispersion: si es alta, el modelo es menos estable entre imagenes.


In [ ]:
# Tabla descriptiva del error cromatico.
error_summary = error_summary.copy()
error_summary["model"] = pd.Categorical(error_summary["model"], categories=model_order, ordered=True)
error_summary = error_summary.sort_values("model")

error_display = error_summary.rename(columns={
    "model": "modelo",
    "n": "n",
    "mean": "media_error",
    "median": "mediana_error",
    "sd": "sd_error",
})
for col in ["media_error", "mediana_error", "sd_error"]:
    error_display[col] = error_display[col].round(2)

display(error_display)

best_mean = error_summary.sort_values("mean").iloc[0]
print(
    f"Menor error medio: {best_mean['model']} "
    f"con media {best_mean['mean']:.2f}."
)


In [ ]:
# Graficos principales del error.
plot_files = [
    "distribucio_error_tots_models.png",
    "violin_boxplot_error_tots_models.png",
]
for name in plot_files:
    path = plots_dir / name
    if path.exists():
        print(path)
        display(DisplayImage(filename=str(path)))
    else:
        print("No encontrado:", path)


### Lazy response

Llamamos **lazy response** a un patron donde el modelo no estima colores concretos, sino que responde con colores muy repetidos o de paleta basica, por ejemplo `00FF00`, `0000FF`, `000000`, `FF00FF` o colores redondeados.

Indicadores usados:

- `unique_hex`: numero de colores distintos que devuelve el modelo;
- `unique_ratio`: proporcion de colores distintos sobre el total de respuestas;
- `max_repeat`: cuantas veces aparece el color mas repetido;
- `top10_repeat_pct`: porcentaje de respuestas concentradas en los 10 colores mas repetidos.

Un modelo con muchas respuestas unicas y bajo `top10_repeat_pct` esta estimando de forma mas fina. Un modelo con pocos colores unicos y alto `top10_repeat_pct` probablemente esta usando respuestas de paleta o aproximaciones demasiado genericas.


In [ ]:
lazy_summary = lazy_summary.copy()
lazy_summary["model"] = pd.Categorical(lazy_summary["model"], categories=model_order, ordered=True)
lazy_summary = lazy_summary.sort_values("model")

lazy_display = lazy_summary.rename(columns={
    "model": "modelo",
    "unique_hex": "colores_unicos",
    "unique_ratio": "ratio_unicos",
    "max_repeat": "max_repeticion",
    "top10_repeat_pct": "top10_repetidos_pct",
})
lazy_display["ratio_unicos"] = lazy_display["ratio_unicos"].round(3)
lazy_display["top10_repetidos_pct"] = lazy_display["top10_repetidos_pct"].round(2)
display(lazy_display)

path = plots_dir / "repeticio_respostes_tots_models.png"
if path.exists():
    print(path)
    display(DisplayImage(filename=str(path)))

worst_lazy = lazy_summary.sort_values("top10_repeat_pct", ascending=False).iloc[0]
print(
    f"Mayor concentracion de respuestas repetidas: {worst_lazy['model']} "
    f"con {worst_lazy['top10_repeat_pct']:.1f}% de respuestas dentro de sus 10 colores mas repetidos."
)


### Relacion entre error y chroma

Aqui comprobamos si los modelos fallan mas en colores poco saturados o muy saturados. Primero calculamos la correlacion entre `error_cromatic` y `chroma` por modelo:

$$
r = corr(error_{ij}, chroma_i)
$$

- Si $r > 0$, el error tiende a aumentar cuando el chroma aumenta.
- Si $r < 0$, el error tiende a disminuir cuando el chroma aumenta.
- Si $r \approx 0$, no hay una relacion lineal clara.

Despues dividimos el chroma en 4 buckets mediante cuartiles. Asi cada grupo contiene aproximadamente el 25% de las imagenes:

- `Q1`: chroma mas bajo;
- `Q2` y `Q3`: chroma intermedio;
- `Q4`: chroma mas alto.

Esto permite ver si un modelo se degrada en una zona concreta de saturacion aunque la correlacion global sea pequena.


In [ ]:
# Correlacion error-chroma y comportamiento por 4 buckets de chroma.
analysis_df = sample_results.dropna(subset=["error_cromatic", "chroma"]).copy()

corr_rows = []
for model, group in analysis_df.groupby("model", observed=False):
    group = group.dropna(subset=["error_cromatic", "chroma"])
    corr = group["error_cromatic"].corr(group["chroma"]) if len(group) > 1 else np.nan
    corr_rows.append({"modelo": str(model), "n": len(group), "corr_error_chroma": corr})

corr_table = pd.DataFrame(corr_rows)
corr_table["corr_error_chroma"] = corr_table["corr_error_chroma"].round(3)
display(corr_table)

# Buckets globales de chroma: se calculan una vez por imagen para que todos los modelos usen los mismos cortes.
image_chroma = analysis_df[["image_name", "chroma"]].drop_duplicates()
image_chroma["chroma_bucket"] = pd.qcut(
    image_chroma["chroma"],
    q=4,
    labels=["Q1 bajo", "Q2 medio-bajo", "Q3 medio-alto", "Q4 alto"],
    duplicates="drop",
)
analysis_df = analysis_df.drop(columns=["chroma_bucket"], errors="ignore").merge(
    image_chroma[["image_name", "chroma_bucket"]],
    on="image_name",
    how="left",
)

bucket_summary = (
    analysis_df.groupby(["model", "chroma_bucket"], observed=False)["error_cromatic"]
    .agg(n="count", media="mean", mediana="median", sd="std")
    .reset_index()
)
bucket_summary["model"] = pd.Categorical(bucket_summary["model"], categories=model_order, ordered=True)
bucket_summary = bucket_summary.sort_values(["model", "chroma_bucket"])
for col in ["media", "mediana", "sd"]:
    bucket_summary[col] = bucket_summary[col].round(2)

display(bucket_summary)

path = plots_dir / "error_vs_chroma_tots_models.png"
if path.exists():
    print(path)
    display(DisplayImage(filename=str(path)))


### Porcentaje de acierto por umbrales

Como el color es continuo, acertar el HEX exacto es una medida demasiado estricta. Por eso usamos varios umbrales de error cromatico:

$$
acierto_t = \frac{1}{n}\sum I(error_{ij} \leq t)
$$

con $t \in \{10, 20, 30, 50\}$. Por ejemplo, `err_le_30_pct` indica el porcentaje de respuestas cuyo error cromatico es como maximo 30 unidades RGB.


In [ ]:
accuracy_summary = accuracy_summary.copy()
accuracy_summary["model"] = pd.Categorical(accuracy_summary["model"], categories=model_order, ordered=True)
accuracy_summary = accuracy_summary.sort_values("model")
accuracy_display = accuracy_summary.rename(columns={
    "model": "modelo",
    "exact_hex_pct": "exacto_hex_pct",
})
for col in accuracy_display.columns:
    if col.endswith("pct"):
        accuracy_display[col] = accuracy_display[col].round(2)
display(accuracy_display)
